# TP – Lubrification non newtonienne (loi puissance)

Ce TP illustre la modélisation numérique des profils de vitesse pour des écoulements en film mince de fluides **loi puissance** (rhéologie non-newtonienne).

Quatre cas sont progressivement traités :
1. Poiseuille avec gradient de pression imposé
2. Poiseuille avec débit imposé
3. Couette–Poiseuille avec débit imposé
4. Passage au fin (par exemple calandrage) avec débit imposé

---
**Comment utiliser ce notebook ?**

- Les cellules marquées `# TODO` contiennent des blocs à compléter.
- Le niveau de guidage est indiqué : ⭐ *guidé*, ⭐⭐ *semi-guidé*, ⭐⭐⭐ *autonome*.
- Ne modifiez pas les cellules sans `# TODO` : elles assurent la cohérence du notebook.
- Exécutez les cellules **dans l'ordre** (Shift+Entrée).


## 1. Import des bibliothèques et paramètres globaux

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq


## 2. Paramètres physiques et domaine de calcul

| Paramètre | Symbole | Valeur | Unité |
|-----------|---------|--------|-------|
| Épaisseur du film | $h$ | 1.0 | m |
| Indice de consistance | $K$ | 150 000 | Pa·s$^n$ |
| Indice loi puissance | $n$ | 0.2 | – |
| Nombre de points de discrétisation | $N_y$ | 500 | – |


In [4]:
h = 1.0
K = 150000.0
n = 0.2
Ny = 500
y = np.linspace(0, h, Ny)


## Fonction utilitaire – préfacteur loi puissance

Pour un fluide loi puissance de consistance $K$ et d'indice $n$, soumis à un gradient de pression $G = dp/dx$, le profil de vitesse Poiseuille fait intervenir le préfacteur :

$$A(G) = \frac{n}{n+1} \cdot \frac{|G|^{1/n-1}\, G}{K^{1/n}}$$

> ⭐ **Question 1 (guidée)** – Complétez la fonction `power_law_prefactor` ci-dessous en vous aidant de la formule ci-dessus.


In [ ]:
def power_law_prefactor(G, K, n):
    """
    Calcule le préfacteur A(G) = (n/(n+1)) * |G|^(1/n-1)*G / K^(1/n).

    Paramètres
    ----------
    G : float   – gradient de pression dp/dx [Pa/m]
    K : float   – indice de consistance [Pa.s^n]
    n : float   – indice loi puissance [-]

    Retour
    ------
    A : float
    """
    # TODO ⭐ : retourner (n / (n+1)) * |G|^(1/n - 1) * G / K^(1/n)
    # Indice : abs(G)**(1/n - 1) * G conserve le signe de G
    return ...  # remplacez "..." par votre expression


## 3. Cas 1 – Poiseuille avec gradient de pression imposé

Le profil analytique de vitesse pour un Poiseuille loi puissance est :

$$u(y) = A(G) \left[ \left|y - \frac{h}{2}\right|^p - \left(\frac{h}{2}\right)^p \right], \quad p = \frac{n+1}{n}$$

avec la condition $u=0$ aux deux parois ($y=0$ et $y=h$).

> ⭐ **Question 2 (guidée)** – Implémentez `velocity_poiseuille_G` en utilisant `power_law_prefactor` et la formule ci-dessus.


In [ ]:
def velocity_poiseuille_G(y, h, G, K, n):
    """
    Profil de vitesse Poiseuille loi puissance avec gradient imposé.

    Paramètres
    ----------
    y : array   – coordonnées [0, h]
    h : float   – épaisseur du film
    G : float   – gradient de pression dp/dx
    K, n        – rhéologie

    Retour
    ------
    u : array   – vitesse u(y)
    """
    A = power_law_prefactor(G, K, n)
    p = (n + 1) / n
    # TODO ⭐ : retourner A * ( |y - h/2|^p - (h/2)^p )
    # Indice : np.abs(y - h/2)**p
    return ...  # remplacez "..." par votre expression


In [ ]:
# --- Application numérique ---
G = -5e5
u1 = velocity_poiseuille_G(y, h, G, K, n)

plt.figure()
plt.plot(u1, y)
plt.xlabel('u(y)')
plt.ylabel('y')
plt.title('Poiseuille – G imposé')
plt.grid()
plt.show()


## 4. Cas 2 – Poiseuille avec débit imposé

Par intégration analytique du profil, le débit vaut :

$$Q = -\frac{2n}{2n+1} \cdot \frac{|G|^{1/n-1}\, G}{K^{1/n}} \cdot \left(\frac{h}{2}\right)^{(2n+1)/n}$$

Cette relation est **inversible analytiquement** : on peut exprimer $G$ en fonction de $Q$.

> ⭐⭐ **Question 3 (semi-guidée)** – Complétez les deux fonctions ci-dessous :
> - `flowrate_poiseuille` calcule $Q$ à partir de $G$ (formule directe).
> - `gradient_from_Q_poiseuille` inverse la relation : posez $Q = C \cdot G$ avec $C$ la constante qui regroupe tous les termes, puis isolez $G$.
>
> *Aide :* l'expression de $C$ est donnée dans la docstring.


In [ ]:
def flowrate_poiseuille(G, h, K, n):
    """
    Débit analytique en Poiseuille loi puissance.
    Q = -(2n)/(2n+1) * |G|^(1/n-1)*G * (h/2)^((2n+1)/n) / K^(1/n)
    """
    # TODO ⭐⭐ : implémenter la formule ci-dessus
    return ...


def gradient_from_Q_poiseuille(Q, h, K, n):
    """
    Gradient de pression obtenu par inversion analytique de flowrate_poiseuille.

    Principe : Q = C * (|G|^(1/n-1) * G)  avec
        C = -(2n)/(2n+1) * (h/2)^((2n+1)/n) / K^(1/n)
    Donc |G|^(1/n) = |Q/C|  => G = -sign(Q) * (|Q|/|C|)^n
    """
    # TODO ⭐⭐ : calculer C, puis retourner G = -np.sign(Q) * (abs(Q)/abs(C))**n
    C = ...  # constante (tous les termes hors G)
    return ...


In [ ]:
# --- Application numérique ---
Q = 1.5
G2 = gradient_from_Q_poiseuille(Q, h, K, n)
u2 = velocity_poiseuille_G(y, h, G2, K, n)

plt.figure()
plt.plot(u2, y)
plt.xlabel('u(y)')
plt.ylabel('y')
plt.title('Poiseuille – Q imposé')
plt.grid()
plt.show()


## 5. Cas 3 – Couette–Poiseuille avec débit imposé

### Rappels théoriques

En présence de parois mobiles ($U_0$ en $y=0$, $U_h$ en $y=h$), le profil loi puissance s'écrit :

$$u(y) = S\,|y + C|^p + C_2$$

où $p = (n+1)/n$, $C$ est la position du point de vitesse nulle du gradient, et :

$$S = A(G), \quad C_2 = U_0 - S\,|C|^p$$

### Stratégie numérique (changement de variable)

Pour des fluides fortement pseudo-plastiques ($n \ll 1$) et $K$ élevé, résoudre directement en $G$ est mal conditionné. On introduit :

$$S = A(G) = \frac{n}{n+1}\frac{|G|^{1/n-1}\,G}{K^{1/n}}$$

et on travaille en $(C, S)$. La méthode de tir procède en 3 étapes :

1. Pour $S$ donné → résoudre $S(|h+C|^p - |C|^p) = U_h - U_0$ pour trouver $C$ **(Brent)**
2. Pour $S$ (et donc $C$) donnés → calculer le débit $Q(S)$ par intégration numérique
3. Trouver $S^*$ tel que $Q(S^*) = Q_{\\text{cible}}$ **(Brent)**

---

> ⭐⭐ **Question 4 (semi-guidée)** – Complétez l'étape 1 (`solve_C_for_S`) : la fonction résiduelle $F(C)$ doit s'annuler pour la bonne valeur de $C$. Écrivez $F(C)$ à partir de la condition aux limites en $y=h$.


In [ ]:
# ============================================================
# ÉTAPE 1 — RÉSOLUTION DE C POUR UN S DONNÉ
# ============================================================

def solve_C_for_S(S):
    """
    Résout l'équation implicite en C :
        S*(|h + C|^p  - |C|^p) = Uh - U0

    Cette équation provient de la condition aux limites u(h) = Uh
    appliquée au profil u(y) = S*|y+C|^p + C2,  avec C2 = U0 - S*|C|^p.

    Utilise la méthode de Brent (scipy.optimize.brentq).
    """
    def F(C):
        # TODO ⭐⭐ : retourner S*(|h+C|^p - |C|^p) - (Uh - U0)
        return ...

    Cmax = 10 * h
    return brentq(F, -Cmax, Cmax)


# ============================================================
# ÉTAPE 2 — CALCUL DU DÉBIT POUR UN S DONNÉ
# ============================================================

def compute_Q_for_S(S):
    """
    Calcule Q(S) par intégration numérique du profil de vitesse.
    """
    C  = solve_C_for_S(S)
    C2 = U0 - S * abs(C)**p
    u  = S * abs(y + C)**p + C2
    # TODO ⭐ : intégrer u sur y avec np.trapz
    return ...


# ============================================================
# ÉTAPE 3 — TIR SUR S (BRENT)
# ============================================================

def find_S_for_Q(Q_target):
    """
    Trouve S* tel que compute_Q_for_S(S*) = Q_target.
    Utilise un encadrement automatique puis brentq.
    """
    Q_couette = 0.5 * (U0 + Uh) * h
    if abs(Q_target - Q_couette) < 1e-12:
        return 0.0   # cas Couette pur : S = 0

    # Encadrement initial
    Smin, Smax = -1.0, 1.0

    # Élargissement automatique jusqu'à encadrement valide
    while (compute_Q_for_S(Smin) - Q_target) * (compute_Q_for_S(Smax) - Q_target) > 0:
        Smin *= 2
        Smax *= 2

    # TODO ⭐⭐ : appeler brentq pour trouver S* dans [Smin, Smax]
    # Indice : la fonction à annuler est  lambda S: compute_Q_for_S(S) - Q_target
    return ...


In [ ]:
# --- Paramètres Couette–Poiseuille ---
U0 = 0.1    # vitesse paroi y=0
Uh = 1.2    # vitesse paroi y=h

Q_target = 1.2   # débit imposé (≠ Couette pur = 0.65)

p = (n + 1) / n  # exposant loi puissance

# --- Résolution ---
S_sol = find_S_for_Q(Q_target)
C_sol = solve_C_for_S(S_sol)

# Profil final
C2_sol = U0 - S_sol * abs(C_sol)**p
u_sol  = S_sol * abs(y + C_sol)**p + C2_sol

print("Solution trouvée :")
print("S =", S_sol)
print("C =", C_sol)
print("u(0) ; u(h) =", u_sol[0], u_sol[-1])
print("Débit =", np.trapz(u_sol, y))

plt.figure()
plt.plot(u_sol, y)
plt.xlabel('u(y)')
plt.ylabel('y')
plt.title('Couette–Poiseuille – Q imposé')
plt.grid()
plt.show()


## 6. Cas 4 – Passage au fin (calandrage) avec débit imposé

On généralise le cas 3 à une **géométrie variable** : l'entrefer $h(x)$ varie le long de la direction d'écoulement (approximation parabolique de deux cylindres de rayons $R_1$ et $R_2$) :

$$h(x) = h_0 + \frac{x^2}{2}\left(\frac{1}{R_1} + \frac{1}{R_2}\right)$$

À chaque position $x$, on applique le même problème Couette–Poiseuille local qu'au Cas 3 (hypothèse film mince).

### Organisation du code

Les fonctions de résolution sont **réécrites avec $h$ en argument** (passage au fin = $h$ variable).

> ⭐⭐⭐ **Question 5 (autonome)** – Complétez `solve_C_for_S`, `compute_Q_for_S` et `find_S_for_Q` avec $h$, $U_0$, $U_h$ comme arguments supplémentaires (les signatures sont fournies). Repartez de votre travail du Cas 3.


In [ ]:
# ============================================================
# PARAMÈTRES PHYSIQUES ET GÉOMÉTRIQUES
# ============================================================

friction  = 1.1
U_lower   = 0.5
U_upper   = U_lower * friction

Q_couette_0 = 0.5 * (U_lower + U_upper) * h
Q_cyl       = 1.2 * Q_couette_0   # débit imposé > Couette pur

p = (n + 1) / n

# Géométrie cylindres (profil parabolique)
R1 = 1.0
R2 = 1.0
h0 = 1.0

L      = 2.0
Nx     = 80
x_cyl  = np.linspace(-L, 0.001, Nx)
h_cyl  = h0 + 0.5 * x_cyl**2 * (1/R1 + 1/R2)

# Visualisation de la géométrie
plt.figure(figsize=(9, 4))
plt.plot(x_cyl, -h_cyl/2, 'k', lw=2)
plt.plot(x_cyl,  h_cyl/2, 'k', lw=2)
plt.xlabel('x')
plt.ylabel('y')
plt.title('Géométrie de la calandre')
plt.axis('equal')
plt.grid()
plt.show()


In [ ]:
# ============================================================
# FONCTIONS DE RÉSOLUTION LOCALE (h, U0, Uh comme arguments)
# ============================================================

def solve_C_for_S(S, h, U0, Uh):
    """
    Résout S*(|h+C|^p - |C|^p) = Uh - U0  pour C.
    Identique au Cas 3, mais h, U0, Uh sont passés en argument.
    """
    # TODO ⭐⭐⭐ : reprendre la fonction du Cas 3 en ajoutant h, U0, Uh
    def F(C):
        return ...
    return brentq(F, -10*h, 10*h)


def compute_Q_for_S(S, h, U0, Uh):
    """
    Calcule Q(S) pour un entrefer h et des parois U0, Uh.
    """
    # TODO ⭐⭐⭐ : reprendre la fonction du Cas 3 avec les nouveaux arguments
    C  = solve_C_for_S(S, h, U0, Uh)
    C2 = ...
    y  = np.linspace(0, h, Ny)
    u  = ...
    return np.trapz(u, y)


def find_S_for_Q(h, U0, Uh, Q_target):
    """
    Tir sur S pour imposer Q(S) = Q_target, avec entrefer h.
    """
    Q_couette = 0.5 * (U0 + Uh) * h
    if abs(Q_target - Q_couette) < 1e-10:
        return 0.0

    Smin, Smax = -1.0, 1.0
    while (compute_Q_for_S(Smin, h, U0, Uh) - Q_target) * \
          (compute_Q_for_S(Smax, h, U0, Uh) - Q_target) > 0:
        Smin *= 2
        Smax *= 2

    # TODO ⭐⭐⭐ : appeler brentq
    return ...


def cyl_velocity_profile(y, h, C, S, n, U_lower, U_upper):
    """
    Profil de vitesse Couette–Poiseuille (loi puissance) en repère physique y ∈ [0, h].

    Paramètres
    ----------
    y            : array  – coordonnée verticale dans [0, h]
    h            : float  – épaisseur locale du film
    C, S         : floats – paramètres issus du tir
    n            : float  – indice loi puissance
    U_lower/upper: floats – vitesses des parois

    Retour
    ------
    u : array – profil u(y)
    """
    p  = (n + 1) / n
    C2 = U_lower - S * abs(C)**p
    return S * abs(y + C)**p + C2


In [ ]:
# ============================================================
# RÉSOLUTION LOCALE À CHAQUE POSITION x
# ============================================================

C_x = np.zeros(Nx)
S_x = np.zeros(Nx)

for i in range(Nx):
    hi       = h_cyl[i]
    S_x[i]  = find_S_for_Q(hi, U_lower, U_upper, Q_cyl)
    C_x[i]  = solve_C_for_S(S_x[i], hi, U_lower, U_upper)


# ============================================================
# CALCUL DE dp/dx ET INTÉGRATION DE LA PRESSION
# ============================================================

# Gradient de pression reconstruit à partir de S
G_x = np.sign(S_x) * ((n+1)/n * np.abs(S_x) * K**(1/n))**n

# Intégration trapèze
p_raw = np.zeros_like(x_cyl)
for i in range(1, len(x_cyl)):
    p_raw[i] = p_raw[i-1] + 0.5*(G_x[i] + G_x[i-1])*(x_cyl[i] - x_cyl[i-1])

# Correction affine pour imposer p=0 aux deux bords
xL, xR = x_cyl[0], x_cyl[-1]
a   = (p_raw[-1] - p_raw[0]) / (xR - xL)
b   = p_raw[0] - a*xL
p_x = p_raw - (a*x_cyl + b)

plt.figure()
plt.plot(x_cyl, p_x)
plt.axhline(0, color='k', linestyle='--', linewidth=1)
plt.xlabel('x')
plt.ylabel('p(x)')
plt.title('Pression dans la calandre (p=0 aux deux bords)')
plt.grid()
plt.show()


In [ ]:
# ============================================================
# CARTOGRAPHIE u(x,y) — TRICONTOURF
# ============================================================

from matplotlib.tri import Triangulation

N_plot  = 10   # afficher un profil tous les N_plot points
X_pts, Y_pts, U_pts = [], [], []
Ny_loc = Ny

for i in range(len(x_cyl)):
    xi = x_cyl[i]
    hi = h_cyl[i]

    y_phys = np.linspace(0.0, hi, Ny_loc)
    u_loc  = cyl_velocity_profile(y_phys, hi, C_x[i], S_x[i], n, U_lower, U_upper)
    y_plot = y_phys - hi/2   # recentrage pour la visu

    if i % N_plot == 0:
        plt.figure(figsize=(9, 4))
        plt.plot(u_loc, y_plot)
        plt.xlabel('Vx')
        plt.ylabel('y')
        plt.title(f'Profil de vitesse à x = {xi:.3f}')
        plt.grid()
        plt.show()

    X_pts.extend(xi * np.ones_like(y_plot))
    Y_pts.extend(y_plot)
    U_pts.extend(u_loc)

X_pts = np.array(X_pts)
Y_pts = np.array(Y_pts)
U_pts = np.array(U_pts)

# Triangulation et masque géométrique
tri  = Triangulation(X_pts, Y_pts)
mask = np.zeros(len(tri.triangles), dtype=bool)
for k, tri_idx in enumerate(tri.triangles):
    x_c   = X_pts[tri_idx].mean()
    y_c   = Y_pts[tri_idx].mean()
    h_loc = np.interp(x_c, x_cyl, h_cyl)
    if not (-h_loc/2 <= y_c <= h_loc/2):
        mask[k] = True
tri.set_mask(mask)

plt.figure(figsize=(9, 4))
plt.plot(x_cyl, -h_cyl/2, 'k', lw=2)
plt.plot(x_cyl,  h_cyl/2, 'k', lw=2)
cont = plt.tricontourf(tri, U_pts, levels=50, cmap='coolwarm')
plt.colorbar(cont, label='u(x,y)')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Cartographie de la vitesse u(x,y)')
plt.grid()
plt.show()


## Pour aller plus loin (optionnel)

> ⭐⭐⭐ **Question 6 – Analyse de sensibilité**
>
> Modifiez l'indice $n$ (essayez 0.1, 0.5, 1.0) et observez l'effet sur :
> - la forme du profil de vitesse (Cas 1)
> - la pression dans la calandre (Cas 4)
>
> Que se passe-t-il quand $n \to 1$ (fluide newtonien) ?

> ⭐⭐⭐ **Question 7 – Force de calandrage**
>
> Calculez et tracez la force normale exercée par le fluide sur les cylindres :
> $$F = \int_{x_{\min}}^{0} p(x)\, dx$$
> *Indice :* utilisez `np.trapz(p_x, x_cyl)`.
